# Exercise 10 — A detector from scratch and hyperparameter optimization

Part 1: train a small YOLO on synthetic shapes and try a pretrained one on your webcam.
Part 2: compare Optuna searches on e-bike trips, then tune a leaf classifier.

**Work without AI agents.** Fill in the `# TODO` blocks and replace the lines marked `# PLACEHOLDER: replace`.

The theory of both parts is in the [theory slides](https://tomasvicar.github.io/MLR-public/exercises/ex10_theory.html) ([PDF](https://tomasvicar.github.io/MLR-public/exercises/ex10_theory.pdf)).

| Task | Your output |
| --- | --- |
| 1. YOLO from scratch | The target tensor, one term of the loss, `box_iou`, `nms`, precision and recall |
| 2. The webcam detector | Answers to three questions about a pretrained YOLO |
| 3. Search on a map | The objective function, three samplers, a range check |
| 4. Tune the leaf network | Your own Optuna study with pruning, and one test on photos of another group |

In [ ]:
%matplotlib inline
import importlib.util
import os
import subprocess
import sys
import time
import urllib.request
import zipfile

if importlib.util.find_spec("optuna") is None:            # Colab does not have Optuna
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

optuna.logging.set_verbosity(optuna.logging.WARNING)    # no log line per trial
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/"
print("torch", torch.__version__, "| optuna", optuna.__version__, "| device:", device)

## Task 1 — YOLO from scratch

**YOLO** (*you only look once*) detects all objects in one forward pass of
one CNN. The image is split into an $S \times S$ grid; every cell predicts
$B$ boxes (each $x, y, w, h$ and a confidence) and $C$ class probabilities,
so the output is $S \times S \times (5B + C)$ (YOLOv1: $7 \times 7 \times 30$).
We build a toy version that trains on a CPU.

### 1a. The scenes and the target tensor

Generated data: noisy grey $96 \times 96$ images with one to three shapes —
**circle, square, triangle** ($C = 3$). The ground truth is one box
$(x_1, y_1, x_2, y_2)$ in pixels and one class per shape. We make sure that
no grid cell holds two object centres (YOLOv1 could not handle that).

In [ ]:
IMG, S, B = 96, 6, 2                       # image size, grid S x S, boxes per cell
CLASSES = ["circle", "square", "triangle"]
C = len(CLASSES)
CELL = IMG // S                            # one cell is 16 x 16 pixels
COLOURS = ["tab:red", "tab:blue", "tab:green"]
yy, xx = np.mgrid[0:IMG, 0:IMG] + 0.5      # pixel centres

images, all_gt_boxes, all_gt_labels = [], [], []
for seed, n_scenes in [(0, 2000), (1, 300)]:            # training scenes, then test scenes
    rng = np.random.default_rng(seed)
    for _ in range(n_scenes):
        image = rng.uniform(0.35, 0.65) + rng.normal(0, 0.06, (IMG, IMG, 3))
        boxes, labels, cells = [], [], []
        for _ in range(rng.integers(1, 4)):
            size = rng.uniform(14, 36)
            cx, cy = rng.uniform(size / 2, IMG - size / 2, 2)
            box = [cx - size / 2, cy - size / 2, cx + size / 2, cy + size / 2]
            overlaps = any(box[0] < b[2] and b[0] < box[2] and box[1] < b[3] and b[1] < box[3] for b in boxes)
            cell = (int(cx // CELL), int(cy // CELL))
            if overlaps or cell in cells:
                continue                   # no overlapping shapes, one object centre per cell
            k = rng.integers(C)
            if k == 0:                     # circle
                mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= (size / 2) ** 2
            elif k == 1:                   # square
                mask = (abs(xx - cx) <= size / 2) & (abs(yy - cy) <= size / 2)
            else:                          # triangle, apex at the top
                frac = (yy - box[1]) / size
                mask = (frac >= 0) & (frac <= 1) & (abs(xx - cx) <= frac * size / 2)
            image[mask] = rng.uniform(0, 1, 3)
            boxes.append(box)
            labels.append(k)
            cells.append(cell)
        images.append(np.clip(image, 0, 1).astype(np.float32))
        all_gt_boxes.append(np.array(boxes))
        all_gt_labels.append(np.array(labels))

images = torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2)   # (n, 3, IMG, IMG)
train_images, train_boxes, train_labels = images[:2000], all_gt_boxes[:2000], all_gt_labels[:2000]
test_images, test_boxes, test_labels = images[2000:], all_gt_boxes[2000:], all_gt_labels[2000:]
print("training scenes", tuple(train_images.shape), "| objects:", sum(len(k) for k in train_labels))
print("test scenes", tuple(test_images.shape), "| objects:", sum(len(k) for k in test_labels))

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i].permute(1, 2, 0))
    for (x1, y1, x2, y2), k in zip(train_boxes[i], train_labels[i]):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=COLOURS[k], lw=1.5))
    ax.set_xticks(range(0, IMG + 1, CELL))                # grid lines at the cell borders
    ax.set_yticks(range(0, IMG + 1, CELL))
    ax.grid(color="white", lw=0.4)
    ax.tick_params(labelbottom=False, labelleft=False, length=0)
    ax.set_title(", ".join(CLASSES[k] for k in train_labels[i]), fontsize=9)
plt.show()

**The target: which cell is responsible and what it must output.** The
**responsible cell** of a box is the cell that contains its centre
$(c_x, c_y)$: column $\lfloor c_x / 16 \rfloor$, row $\lfloor c_y / 16 \rfloor$.
In that cell, store

- $x, y$ — the centre **relative to the cell**, in $[0, 1)$: $x = c_x / 16 - \text{column}$;
- $w, h$ — the size **relative to the image**: $w = (x_2 - x_1) / 96$;
- an object flag of $1$ and the **one-hot class**.

All other cells are zeros. The target has one box per cell:
$6 \times 6 \times (5 + C) = 6 \times 6 \times 8$. (The network will output
$B = 2$ boxes per cell: $6 \times 6 \times 13$.)

**By hand first:** a circle with the box $(40, 20, 64, 52)$. Which row and
column, and which eight numbers?

Then fill in the loop. The last axis is
`[x, y, w, h, object flag, circle, square, triangle]`. Scene 0 is the by-hand
circle, so compare its printed target with your by-hand answer.

In [ ]:
scenes = [(np.array([[40, 20, 64, 52]]), np.array([0]))] + list(zip(train_boxes, train_labels))
targets = torch.zeros(len(scenes), S, S, 5 + C)
for n, (boxes, labels) in enumerate(scenes):
    for (x1, y1, x2, y2), k in zip(boxes, labels):
        # Centre -> responsible row and column; x, y relative to the cell; w, h relative
        # to the image; confidence 1; a 1 at the position of the class.
        # TODO: your code here
        ...
example_target, train_targets = targets[0], targets[1:]

# Check: the by-hand circle, and one non-zero cell per object.
print("non-zero cell (row, column):", torch.nonzero(example_target[..., 4]).tolist())
print("its eight numbers:", np.round(example_target[example_target[..., 4] == 1].numpy(), 3))
assert train_targets[..., 4].sum() == sum(len(k) for k in train_labels), "every object must land in its own cell"
print("targets", tuple(train_targets.shape))

### 1b. The network and the YOLO loss

A small fully convolutional CNN: four blocks (convolution, batch norm, leaky
ReLU, max pooling) shrink $96 \to 48 \to 24 \to 12 \to 6$, two more
convolutions widen the view, and a $1 \times 1$ convolution gives
$5B + C = 13$ numbers in each of the $6 \times 6$ cells.

In [ ]:
torch.manual_seed(0)                                   # run this cell again to train from scratch
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.LeakyReLU(0.1), nn.MaxPool2d(2),     # 48 x 48
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.1), nn.MaxPool2d(2),    # 24 x 24
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1), nn.MaxPool2d(2),    # 12 x 12
    nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1), nn.MaxPool2d(2),    # 6 x 6
    nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1),
    nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1),
    nn.Conv2d(128, 5 * B + C, 1))                                                               # 13 numbers per cell
print("parameters:", sum(p.numel() for p in model.parameters()))
out = model(train_images[:2]).permute(0, 2, 3, 1)      # (N, S, S, 5B + C): the cells, then the numbers
print("output for 2 images:", tuple(out.shape))

**From raw numbers to boxes.** A **sigmoid** maps $x, y$, the confidence and
$\sqrt{w}, \sqrt{h}$ into $[0, 1]$; a **softmax** gives the class
probabilities. (YOLOv1 predicts $\sqrt{w}$, so that an error of 2 pixels
costs more on a small box than on a large one.)

**The loss** is the YOLOv1 sum of squared errors. Box $j$ of cell $i$ is
**responsible** ($\mathbb 1^{\text{obj}}_{ij} = 1$) when the cell holds an
object centre and box $j$ has the larger IoU of the $B$ boxes with the ground
truth; $\mathbb 1^{\text{noobj}}_{ij} = 1 - \mathbb 1^{\text{obj}}_{ij}$.
The hat marks the target:

$$
\begin{aligned}
L = {} & \lambda_\text{coord} \sum_{i,j} \mathbb 1^{\text{obj}}_{ij} \left[(x_{ij} - \hat x_i)^2 + (y_{ij} - \hat y_i)^2 + (\sqrt{w_{ij}} - \sqrt{\hat w_i})^2 + (\sqrt{h_{ij}} - \sqrt{\hat h_i})^2\right] \\
& + \sum_{i,j} \mathbb 1^{\text{obj}}_{ij} \left(c_{ij} - \mathrm{IoU}_{ij}\right)^2
  + \lambda_\text{noobj} \sum_{i,j} \mathbb 1^{\text{noobj}}_{ij}\, c_{ij}^2
  + \sum_{i \in \text{obj}} \sum_{k=1}^{C} \left(p_{ik} - \hat p_{ik}\right)^2
\end{aligned}
$$

- The **confidence** of the responsible box is trained to its **IoU** with
  the ground truth: "there is an object *and* my box fits it".
- Almost all of the $6 \cdot 6 \cdot 2 = 72$ boxes are empty, so
  **$\lambda_\text{noobj} = 0.5$** turns their term down and
  **$\lambda_\text{coord} = 5$** turns the box terms up.
- The class term counts only in cells with an object.

**Write the no-object term**: the squared confidence of every box that is
not responsible, summed and divided by the batch size like the other terms.

In [ ]:
def to_corners(xy, wh):
    """Centre relative to the cell and size relative to the image (..., S, S, B, 2) -> x1 y1 x2 y2 in pixels."""
    cols = torch.arange(S, device=xy.device).reshape(1, 1, S, 1)
    rows = torch.arange(S, device=xy.device).reshape(1, S, 1, 1)
    cx = (xy[..., 0] + cols) * CELL
    cy = (xy[..., 1] + rows) * CELL
    w, h = wh[..., 0] * IMG, wh[..., 1] * IMG
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)


def yolo_loss(out, target):
    """The four terms of the YOLOv1 loss. out: (N, S, S, 5B + C) raw, target: (N, S, S, 5 + C)."""
    n = len(out)
    boxes = torch.sigmoid(out[..., :5 * B]).reshape(n, S, S, B, 5)
    xy, sqrt_wh, conf = boxes[..., :2], boxes[..., 2:4], boxes[..., 4]
    probs = torch.softmax(out[..., 5 * B:], dim=-1)
    has_object = target[..., 4]                                        # (N, S, S): 1 in cells with an object
    true_xy, true_wh = target[..., None, :2], target[..., None, 2:4]   # None: the same true box for all B

    # IoU of each predicted box with the true box of its cell (a target, so no gradient)
    pred, true = to_corners(xy, sqrt_wh ** 2), to_corners(true_xy, true_wh)
    top_left = torch.maximum(pred[..., :2], true[..., :2])
    bottom_right = torch.minimum(pred[..., 2:], true[..., 2:])
    inter = (bottom_right - top_left).clamp(min=0).prod(-1)
    area_pred = (pred[..., 2] - pred[..., 0]) * (pred[..., 3] - pred[..., 1])
    area_true = (true[..., 2] - true[..., 0]) * (true[..., 3] - true[..., 1])
    iou = (inter / (area_pred + area_true - inter + 1e-9)).detach()   # (N, S, S, B)

    # Responsible: the box with the larger IoU, only in cells with an object. (N, S, S, B)
    responsible = nn.functional.one_hot(iou.argmax(-1), B) * has_object[..., None]
    box_error = ((xy - true_xy) ** 2 + (sqrt_wh - true_wh.sqrt()) ** 2).sum(-1)
    terms = {
        "coord": (responsible * box_error).sum() / n,
        "conf_obj": (responsible * (conf - iou) ** 2).sum() / n,
        "class": (has_object[..., None] * (probs - target[..., 5:]) ** 2).sum() / n,
    }
    # The no-object confidence term, divided by n like the others.
    # TODO: your code here
    terms["conf_noobj"] = out.new_tensor(0.0)  # PLACEHOLDER: replace
    return terms


# Check: raw zeros give every confidence sigmoid(0) = 0.5; with one object,
# 71 of the 72 boxes are not responsible: 71 * 0.5^2.
terms = yolo_loss(torch.zeros(1, S, S, 5 * B + C), example_target[None])
print({name: round(value.item(), 4) for name, value in terms.items()})
assert abs(terms["conf_noobj"].item() - 71 * 0.25) < 1e-4, "the no-object term should be 71 * 0.25 = 17.75"

### 1c. Training

A standard loop: Adam, batches of 64, a one-cycle learning rate (it rises,
then falls close to zero), and the four terms weighted with
$\lambda_\text{coord} = 5$ and $\lambda_\text{noobj} = 0.5$. Start it and read on.

In [ ]:
EPOCHS, BATCH = 15, 64
torch.manual_seed(0)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
steps = EPOCHS * ((len(train_images) + BATCH - 1) // BATCH)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=2e-3, total_steps=steps)
history = []
start = time.time()
for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(train_images))
    loss_sum = 0.0
    for i in range(0, len(order), BATCH):
        batch = order[i:i + BATCH]
        out = model(train_images[batch].to(device)).permute(0, 2, 3, 1)
        t = yolo_loss(out, train_targets[batch].to(device))
        loss = 5.0 * t["coord"] + t["conf_obj"] + 0.5 * t["conf_noobj"] + t["class"]
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        loss_sum += loss.item() * len(batch)
    history.append(loss_sum / len(order))
    print(f"epoch {epoch + 1:2d}  loss {history[-1]:.3f}  ({time.time() - start:.0f} s)")
model = model.cpu().eval()

plt.figure(figsize=(4, 2.5))
plt.plot(range(1, EPOCHS + 1), history, "o-", ms=3)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.show()

### 1d. Decoding, IoU and NMS

**Decoding** turns the $6 \times 6 \times 13$ output back into boxes: each
cell gives $B = 2$ boxes in pixels, each with its class (the most probable
one) and the score $c \cdot \max_k p_k$ (confidence × that probability).
That is 72 candidates per image, most of them with a score near zero.

In [ ]:
with torch.no_grad():
    out = model(test_images).permute(0, 2, 3, 1)       # (300, S, S, 13)
n = len(out)
boxes = torch.sigmoid(out[..., :5 * B]).reshape(n, S, S, B, 5)
probs = torch.softmax(out[..., 5 * B:], dim=-1)
best_prob, best_class = probs.max(dim=-1)              # (n, S, S)
all_boxes = to_corners(boxes[..., :2], boxes[..., 2:4] ** 2).reshape(n, S * S * B, 4)
all_scores = (boxes[..., 4] * best_prob[..., None]).reshape(n, S * S * B)
all_classes = best_class[..., None].expand(n, S, S, B).reshape(n, S * S * B)
print("candidates per image:", all_boxes.shape[1], "| above score 0.1 on average:",
      round((all_scores > 0.1).sum(1).float().mean().item(), 1))

**IoU** (intersection over union) of two boxes $A$, $B$:

$$
\mathrm{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}.
$$

The intersection is a rectangle: left edge = the larger of the two left
edges, right edge = the smaller of the two right edges (the same for top and
bottom); its width or height is zero when the boxes do not overlap.
Write `box_iou` for two boxes $(x_1, y_1, x_2, y_2)$. Check (the lecture's
example): $A = (2, 2, 8, 6)$, $B = (4, 3, 10, 8)$, IoU $= 12/42 = 0.29$.

In [ ]:
def box_iou(a, b):
    """IoU of two boxes (x1, y1, x2, y2)."""
    # Width and height of the intersection (zero if no overlap), its area,
    # the union, and their ratio.
    # TODO: your code here
    return 0.0  # PLACEHOLDER: replace


print("IoU of the lecture's boxes:", round(box_iou((2, 2, 8, 6), (4, 3, 10, 8)), 4), "= 12/42 =", round(12 / 42, 4))
assert abs(box_iou((2, 2, 8, 6), (4, 3, 10, 8)) - 12 / 42) < 1e-9
assert box_iou((0, 0, 1, 1), (2, 2, 3, 3)) == 0, "boxes that do not overlap have IoU 0"
theirs = torchvision.ops.box_iou(all_boxes[0, :5], all_boxes[0, :5])     # 5 decoded boxes, all pairs
for i in range(5):
    for j in range(5):
        assert abs(box_iou(all_boxes[0, i].tolist(), all_boxes[0, j].tolist()) - theirs[i, j].item()) < 1e-5, "differs from torchvision.ops.box_iou"
print("matches torchvision.ops.box_iou")

**NMS** (non-maximum suppression) removes duplicates: neighbouring cells and
both boxes of a cell often fire on the same object. Sort by score, **keep**
the best box, **remove** every remaining box with IoU above 0.5 with it,
repeat with the best of the rest. Return the kept indices, best first (as
`torchvision.ops.nms` does).

Check (the lecture's example): scores 0.90, 0.80, 0.70, 0.60 → 0.90 kept,
0.80 removed (IoU 0.68), 0.70 kept (0.33 with the first), 0.60 removed (0.52
with the 0.70 box).

In [ ]:
def nms(boxes, scores, threshold=0.5):
    """Greedy NMS. boxes: a list of (x1, y1, x2, y2), scores: a list of numbers -> the kept indices, best first."""
    # Sort the indices by decreasing score. Repeat: keep the best remaining one,
    # drop the remaining boxes whose IoU with it is above the threshold.
    # TODO: your code here
    return list(range(len(scores)))  # PLACEHOLDER: replace


lecture_boxes = [(10, 10, 30, 30), (12, 12, 32, 32), (20, 10, 40, 30), (23, 14, 43, 34)]
lecture_scores = [0.90, 0.80, 0.70, 0.60]
print("kept of the lecture's example:", nms(lecture_boxes, lecture_scores))
assert nms(lecture_boxes, lecture_scores) == [0, 2]
for i in range(20):                                    # against torchvision on 20 test images
    keep = all_scores[i] > 0.05
    mine = nms(all_boxes[i][keep].tolist(), all_scores[i][keep].tolist())
    theirs = torchvision.ops.nms(all_boxes[i][keep], all_scores[i][keep], 0.5).tolist()
    assert mine == theirs, (i, mine, theirs)
print("matches torchvision.ops.nms on 20 test images")

The detector on unseen test scenes: all candidates with score above 0.1
(top) and what NMS keeps (bottom).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5.6))
for i in range(5):
    keep = all_scores[i] > 0.1
    boxes, scores, classes = all_boxes[i][keep], all_scores[i][keep], all_classes[i][keep]
    kept = nms(boxes.tolist(), scores.tolist())
    for row, idx in enumerate([list(range(len(boxes))), kept]):   # top: all candidates, bottom: after NMS
        ax = axes[row, i]
        ax.imshow(test_images[i].permute(1, 2, 0))
        for j in idx:
            x1, y1, x2, y2 = boxes[j].tolist()
            ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=COLOURS[classes[j]], lw=1.5))
            ax.text(x1, y1 - 1, f"{scores[j]:.2f}", color=COLOURS[classes[j]], fontsize=7)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"{['without', 'with'][row]} NMS: {len(idx)} boxes ({len(test_labels[i])} objects)", fontsize=9)
plt.tight_layout()
plt.show()

### 1e. The confidence threshold: precision and recall

A detection is a **true positive** (TP) if it has the right class and IoU
$\ge 0.5$ with a ground-truth box not yet taken by a better detection;
otherwise it is a **false positive** (FP). A ground-truth box that nothing
found is a **false negative** (FN). As in the second lab,

$$
\text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{TP + FN}.
$$

The loop counts TP, FP and FN on all 300 test scenes for the score
thresholds 0.05, 0.3 and 0.8 with NMS, and 0.05 without NMS. Add precision
and recall. If no detections survive, report precision as NaN. Before you run it: which way will each of them move?

In [ ]:
rows = []
for threshold, use_nms in [(0.05, True), (0.3, True), (0.8, True), (0.05, False)]:
    tp = fp = fn = 0
    for i in range(len(test_images)):
        keep = all_scores[i] >= threshold
        boxes, scores, classes = all_boxes[i][keep].tolist(), all_scores[i][keep].tolist(), all_classes[i][keep].tolist()
        order = nms(boxes, scores) if use_nms else sorted(range(len(scores)), key=lambda j: -scores[j])
        taken = set()                                  # ground-truth boxes already matched
        for j in order:                                # best detection first
            free = [g for g in range(len(test_boxes[i])) if g not in taken and test_labels[i][g] == classes[j]]
            ious = [box_iou(boxes[j], test_boxes[i][g]) for g in free]
            if ious and max(ious) >= 0.5:
                taken.add(free[int(np.argmax(ious))])
                tp += 1
            else:
                fp += 1
        fn += len(test_boxes[i]) - len(taken)
    # TODO: your code here
    precision = recall = 0.0  # PLACEHOLDER: replace
    rows.append({"threshold": threshold, "NMS": use_nms, "TP": tp, "FP": fp, "FN": fn,
                 "precision": precision, "recall": recall})
print(pd.DataFrame(rows).round(3).to_string(index=False))

*Which threshold would you choose for a screening tool that must not miss a
lesion, and which for counting objects? What does NMS change in this table?*

## Task 2 — A pretrained YOLO on your webcam

The [**live detector of the course**](https://tomasvicar.github.io/mlr-detector/)
runs **YOLO11n**, a current YOLO pretrained on the 80 COCO classes, in your
browser (the camera image stays on your computer). It has a **confidence
slider**, an **NMS switch** with an **IoU threshold** slider, and per-class or
class-agnostic NMS (ours in 1d is class-agnostic). Point it at your desk,
your hand, a bottle, a phone, a person.

1. Switch **NMS off**. How many overlapping boxes can you see on one object? Why does
   this detector start with more candidates per image than our 72? Switch it on and move the IoU threshold: when do
   duplicates come back, when does one of two neighbouring objects lose its box?
2. **Freeze** a frame and move the **confidence slider** from low to high.
   Count TP, FP and FN by eye at three settings — which way do precision and
   recall move?
3. Find an object the detector misses or mislabels. Is it a class COCO does
   not have, an unusual view, or too small?

*Your answers:*

## Task 3 — Hyperparameter optimization: the search on a map

Each **trial** scores one hyperparameter setting. Here the SVM uses five
CV fits per trial. Grid and random search ignore earlier scores.
**TPE** starts with random trials and then uses the scores so far to choose
the next settings. In Task 4, a **pruner** also stops bad training runs early.


### 3a. The map: cross-validation scores on a grid

The e-bikes of lab 6 again (the same training trips and cleaning), now with a
scaler and an **RBF SVM**, which has two hyperparameters: $C$ and $\gamma$. The map
shows the 5-fold CV accuracy for $29 \times 29$ combinations, computed in advance
on the same folds (the hidden cell shows how; it takes about 7 minutes on a Colab CPU). We plot each sampler's trials on top of it.

In [ ]:
NUMERIC = ["capacity_wh", "charge_pct", "battery_age_years", "distance_km",
           "ascent_m", "temperature_c", "headwind_kmh", "payload_kg"]
CATEGORICAL = ["surface", "bike_type", "assistance"]
ebikes = pd.read_csv(DATA + "ex06_ebikes.csv")
X_train, X_test, y_train, y_test = train_test_split(
    ebikes[NUMERIC + CATEGORICAL], ebikes["runs_out"], test_size=0.25, stratify=ebikes["runs_out"], random_state=42)

# The cleaning of lab 6: impossible readings become missing (a fixed rule, it learns nothing).
X_train = X_train.copy()
X_train.loc[~X_train["charge_pct"].between(0, 100), "charge_pct"] = np.nan
X_train.loc[X_train["distance_km"] <= 0, "distance_km"] = np.nan

# Imputation, scaling and one-hot are fitted inside every CV fold.
numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
svm = Pipeline([("prep", ColumnTransformer([("num", numeric, NUMERIC), ("cat", categorical, CATEGORICAL)])),
                ("model", SVC(kernel="rbf"))])
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# One fixed setting: fit on four folds, evaluate on the fifth, repeat five times.
svm.set_params(model__C=1.0, model__gamma=0.01)
scores = cross_val_score(svm, X_train, y_train, cv=CV)
print("Fixed parameters: C = 1.0, gamma = 0.01")
print("Accuracy in each fold:", scores.round(3))
print(f"Mean CV accuracy: {scores.mean():.3f}")
svm.set_params(model__C=1.0, model__gamma="scale")  # restore SVC defaults

In [ ]:
#@title Load the SVM validation map (computed in advance)
# The same pipeline, the same folds CV, a 29 x 29 log grid of C and gamma: 4,205 fits,
# about 7 minutes on a Colab CPU, so the map is loaded. The commented code below computed it.
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve("https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/ex10_svm_map.npz", "data/ex10_svm_map.npz")
svm_map = dict(np.load("data/ex10_svm_map.npz"))   # cv[i, j]: CV accuracy at gamma[i], C[j]

# To recompute the map, uncomment (same folds, parallel over the settings):
# from sklearn.model_selection import GridSearchCV
#
# log_C = np.linspace(-2, 4, 29)
# log_gamma = np.linspace(-4, 2, 29)
# map_search = GridSearchCV(svm, {"model__C": 10 ** log_C, "model__gamma": 10 ** log_gamma},
#                           cv=CV, n_jobs=-1, refit=False)
# map_search.fit(X_train, y_train)
# # (GridSearchCV lists C first, then gamma; the plot uses gamma as rows and C as columns.)
# svm_map = {"log10_C": log_C, "log10_gamma": log_gamma,
#            "cv": map_search.cv_results_["mean_test_score"].reshape(len(log_C), len(log_gamma)).T,
#            "default_cv": cross_val_score(svm, X_train, y_train, cv=CV).mean(),
#            "majority": y_train.value_counts(normalize=True).max()}

print("training trips", len(y_train), "| majority class", round(float(svm_map["majority"]), 3),
      "| SVC() defaults", round(float(svm_map["default_cv"]), 3), "| best of the map", round(float(svm_map["cv"].max()), 3))


def plot_map(studies, title):
    """The CV map, with the trials of each study {name: study} on top."""
    plt.figure(figsize=(6.2, 4.8))
    plt.pcolormesh(10 ** svm_map["log10_C"], 10 ** svm_map["log10_gamma"], svm_map["cv"],
                   cmap="viridis", vmin=0.6, vmax=0.94, shading="nearest")
    plt.colorbar(label="5-fold CV accuracy")
    for (name, study), marker in zip(studies.items(), ["s", "o", "^"]):
        plt.scatter([t.params["C"] for t in study.trials], [t.params["gamma"] for t in study.trials], marker=marker,
                    s=28, edgecolor="white", linewidth=0.8, label=f"{name}: best {study.best_value:.3f}")
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("C")
    plt.ylabel("gamma")
    plt.title(title)
    if studies:
        plt.legend(fontsize=8, loc="lower left")
    plt.show()


plot_map({}, "E-bike SVM: 5-fold CV accuracy")

**The objective is a function.** Optuna calls it with one argument, the
`trial`. Inside it, `trial.suggest_float(...)` picks this trial's value of a
hyperparameter; the function then scores the model and returns the number
that Optuna maximizes. The model and the data are the variables defined
above. This example tunes only $C$, with five trials. (Five trials are still TPE's random
start — ten by default — so this cell shows the interface, not the search.)

In [ ]:
def objective(trial):
    C = trial.suggest_float("C", 1e-2, 1e4, log=True)     # the value of C for this trial
    svm.set_params(model__C=C)
    return cross_val_score(svm, X_train, y_train, cv=CV).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=5)
print(study.best_params, round(study.best_value, 3))

### 3b. Three samplers, the same budget

Write `svm_objective(trial)`, which tunes $C$ and $\gamma$:

- suggest `C` in $[10^{-2}, 10^{4}]$ and `gamma` in $[10^{-4}, 10^{2}]$, both with `log=True`;
- set the pipeline parameters `model__C` and `model__gamma`;
- return mean cross-validation accuracy using the fixed folds `CV`.

Then run **25 trials** with each of the three samplers. The grid sampler
tries the 5 × 5 = 25 combinations in `grid`.

In [ ]:
def svm_objective(trial):
    # Suggest C and gamma in their ranges on a log scale; return the mean CV accuracy.
    # TODO: your code here
    return 0.0  # PLACEHOLDER: replace


grid = {"C": list(np.logspace(-2, 4, 5)), "gamma": list(np.logspace(-4, 2, 5))}
samplers = {"grid": optuna.samplers.GridSampler(grid, seed=0),
            "random": optuna.samplers.RandomSampler(seed=0),
            "TPE": optuna.samplers.TPESampler(seed=0)}
studies = {}
for name in samplers:
    # A study (direction "maximize") with this sampler, 25 trials of svm_objective.
    # TODO: your code here
    ...
plot_map(studies, "25 trials each, log ranges")

### 3c. The same search on a linear scale

The cell below repeats random and TPE with the same ranges on a linear scale (`log=False`).
**Predict first** where the random points will land on the map, then run it.

In [ ]:
# Given: the same objective on linear ranges; random and TPE (seed 0), 25 trials each.
def svm_objective_linear(trial):
    C = trial.suggest_float("C", 1e-2, 1e4, log=False)
    gamma = trial.suggest_float("gamma", 1e-4, 1e2, log=False)
    svm.set_params(model__C=C, model__gamma=gamma)
    return cross_val_score(svm, X_train, y_train, cv=CV).mean()


linear = {}
for name, sampler in [("random", optuna.samplers.RandomSampler(seed=0)), ("TPE", optuna.samplers.TPESampler(seed=0))]:
    linear[name] = optuna.create_study(direction="maximize", sampler=sampler)
    linear[name].optimize(svm_objective_linear, n_trials=25)
plot_map(linear, "25 trials each, linear ranges")

*What fraction of a linear range $[10^{-4}, 10^{2}]$ lies above
$\gamma = 1$? What does that mean for where the trials go?*

### 3d. The best value at the edge

Now $C$ is searched only in $[10^{-2}, 10^{-1}]$. Optuna returns a best value
without any warning, although the map shows higher scores outside the range.

In [ ]:
def svm_objective_small_C(trial):
    C = trial.suggest_float("C", 1e-2, 1e-1, log=True)     # only [0.01, 0.1]
    gamma = trial.suggest_float("gamma", 1e-4, 1e2, log=True)
    svm.set_params(model__C=C, model__gamma=gamma)
    return cross_val_score(svm, X_train, y_train, cv=CV).mean()


small = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
small.optimize(svm_objective_small_C, n_trials=25)
print(f"C in [0.01, 0.1]: best {small.best_value:.3f} at {small.best_params}")
# Always check the best values against the ranges you chose. A best value close to an end
# of its range (here C close to 0.1) means the optimum may lie outside: widen the range and run again.

### 3e. A minimal pruning example

This example uses **made-up scores**, no model or data. The score grows
faster for some settings than others. `MedianPruner` can stop a slow trial
before all eight epochs finish.

At each epoch, **report** the score and its step, **ask** whether to stop,
and **raise** `optuna.TrialPruned` if requested. Reporting alone does not
stop a trial. Run the cell and inspect the `COMPLETE` and `PRUNED` states.

In [ ]:
def dummy_objective(trial):
    speed = trial.suggest_float("speed", 0.05, 0.5)
    for epoch in range(8):
        score = 1 - np.exp(-speed * (epoch + 1))  # made-up learning curve
        trial.report(score, step=epoch)           # save this epoch's score
        if trial.should_prune():                 # ask the study's pruner
            print(f"Trial {trial.number}: pruned at epoch {epoch + 1}")
            raise optuna.TrialPruned()            # stop this trial, continue the study
    return score


dummy_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.RandomSampler(seed=0),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=4))
dummy_study.optimize(dummy_objective, n_trials=6)
print(dummy_study.trials_dataframe()[["number", "value", "state", "params_speed"]].round(3).to_string(index=False))

## Task 4 — Tune the leaf network

Now an objective that costs real training: lab 8's leaf network
`LeafNetBN`, trained from scratch on this year's photos. Lab 8's finished
training code is given; you write only the **optimization around it**.

**Data:** the photos of all four lab groups, $96 \times 96$ px, in one zip as
`leaves/group<G>/<train|val>/<class>/`.

- **Tuning:** groups 1–3, `train` to train and `val` to compare the trials.
- **Test:** all of **group 4** (other students, other phones, another day),
  used **once**, at the end.

Group 1 looks different (greener leaves, cluttered backgrounds, hands, a few
photos in the wrong class). A photo that several groups shared is kept once,
in the earliest group, so the test holds no copies of tuning photos.

In [ ]:
if not os.path.exists("data/leaves/files.csv"):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(DATA + "ex10_leaves_2026.zip", "data/ex10_leaves_2026.zip")
    zipfile.ZipFile("data/ex10_leaves_2026.zip").extractall("data")
files = pd.read_csv("data/leaves/files.csv")
print(pd.crosstab([files["group"], files["split"]], files["class"], margins=True))

In [ ]:
train_files = files[(files["group"] <= 3) & (files["split"] == "train")]
val_files = files[(files["group"] <= 3) & (files["split"] == "val")]
test_files = files[files["group"] == 4]
LEAF_CLASSES = sorted(train_files["class"].unique())


class LeafDataset(Dataset):
    """The photos listed in a table (columns file, class)."""
    def __init__(self, table, transform):
        self.paths = ["data/" + f for f in table["file"]]
        self.labels = [LEAF_CLASSES.index(c) for c in table["class"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


class LeafNetBN(nn.Module):
    """Lab 8's LeafNetBN with one new option: dropout before the last layer."""
    def __init__(self, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 6 * 6, 64)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 4)
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(64)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = self.pool(torch.relu(self.bn4(self.conv4(x))))
        return self.fc2(self.drop(torch.relu(self.fc1(x.flatten(1)))))


def accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            correct += (model(images.to(device)).argmax(1).cpu() == labels).sum().item()
    return correct / len(loader.dataset)


mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
plain = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
# Own generators, so that evaluating does not use up the random numbers of the training.
val_loader = DataLoader(LeafDataset(val_files, plain), batch_size=64, generator=torch.Generator())
test_loader = DataLoader(LeafDataset(test_files, plain), batch_size=64, generator=torch.Generator())
DEFAULT = {"lr": 1e-3, "weight_decay": 1e-2, "dropout": 0.0, "min_scale": 0.6, "batch_size": 32}   # lab 8
print(LEAF_CLASSES, "| training", len(train_files), "| validation", len(val_files), "| test", len(test_files))

In [ ]:
EPOCHS = 12

torch.manual_seed(0)
model = LeafNetBN(0.0).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
augment = transforms.Compose([
    transforms.RandomResizedCrop(96, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
train_loader = DataLoader(LeafDataset(train_files, augment), batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(images.to(device)), labels.to(device))
        loss.backward()
        optimizer.step()
print(f"Baseline validation accuracy: {accuracy(model, val_loader):.3f}")

### 4a. Your study

The cell above trains once with fixed parameters. Turn it into an Optuna
objective:

- **Wrap the training** in `leaf_objective(trial)`: create a fresh model
  and optimizer each time and return validation accuracy. Replace
  `torch.manual_seed(0)` with `torch.manual_seed(seed)`, so that the optional
  seeds in 4b really differ.
- **Replace the fixed values in place** with `trial.suggest_…`:
  learning rate in $[10^{-4}, 10^{-2}]$, weight decay in $[10^{-5}, 10^{-1}]$,
  dropout in $[0, 0.5]$, minimum crop area in $[0.3, 1]$, batch size
  one of 16, 32, 64. Use the parameter names in `DEFAULT`. Choose log or
  linear scales yourself.
- **Pruning:** after each epoch, measure validation accuracy and use
  `trial.report`, `trial.should_prune()` and `raise optuna.TrialPruned`,
  as in 3e. Use the zero-based epoch as the step.
- **Study:** TPE with `seed=MY_SEED` (your own number, e.g. the last two digits
  of your student ID, so the room compares different studies) and
  `n_startup_trials=4`; copy the `MedianPruner` settings from 3e.
  Enqueue `DEFAULT` before `optimize`, so **trial 0 is the baseline**
  (lab 8's settings with AdamW, 12 epochs). At least 10 trials,
  15–20 if time allows.

In [ ]:
def leaf_objective(trial, seed=0, return_model=False):
    # Copy the fixed training above, replacing the five settings with suggestions.
    # Report validation accuracy after each epoch and stop if the pruner asks.
    # TODO: your code here
    model = None  # PLACEHOLDER: replace with the training code
    val_accuracy = np.nan  # PLACEHOLDER: compute after each epoch
    if return_model:  # used only for the final test, not during the study
        return model
    return val_accuracy


N_TRIALS = 10   # more if you have the time
MY_SEED = 0     # your own number, e.g. the last two digits of your student ID
# The study: TPE (seed MY_SEED), the median pruner, DEFAULT enqueued as trial 0, N_TRIALS trials.
# TODO: your code here
leaf_study = None  # PLACEHOLDER: replace

table = leaf_study.trials_dataframe()
table["seconds"] = table["duration"].dt.total_seconds().round()
print(table[["number", "value", "state", "seconds", "params_lr", "params_weight_decay",
             "params_dropout", "params_min_scale", "params_batch_size"]].round(4).to_string(index=False))
print("pruned:", (table["state"] == "PRUNED").sum(), "of", len(table))
print(f"best {leaf_study.best_value:.3f} (trial {leaf_study.best_trial.number}), baseline {leaf_study.trials[0].value:.3f}")
print("best parameters:", leaf_study.best_params)    # check them against their ranges, as in 3d

### 4b. The test, once

Before looking at group 4, decide whether to include the optional seeds 1 and 2.
Keep both configurations fixed for the whole comparison.

`optuna.trial.FixedTrial(params)` supplies fixed values to the same
`trial.suggest_…` calls; it does not search or prune. Call
`leaf_objective(optuna.trial.FixedTrial(params), seed=0, return_model=True)`
to train with `DEFAULT` and then the best parameters. The `return_model`
switch gives you the trained model for evaluation.

Record validation and test accuracy in a DataFrame called `results` with
columns `configuration`, `seed`, `validation` and `test (group 4)`.

In [ ]:
# For DEFAULT and the best parameters: train with seed 0, then the validation and the test accuracy.
# TODO: your code here
results = None  # PLACEHOLDER: replace
print(results.round(3).to_string(index=False))

**Optional:** repeat both configurations with seeds 1 and 2. How much does the
test accuracy of one configuration move between seeds, compared with the gap
between the two configurations?

In [ ]:
# Optional: seeds 1 and 2 for both configurations; print all runs and the means.
# TODO: your code here
...

*How much accuracy is one validation photo worth? Is the gain of the best
trial over the baseline larger than that? Why is the best validation accuracy
of a study optimistic, and how much of the gain is left on group 4?*